In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import seaborn as sns

from keras.callbacks import EarlyStopping
from keras.callbacks import ReduceLROnPlateau
from keras import backend as K
import gc
import os
import time

In [2]:
Yt = pd.read_csv('ws.csv',header=1,parse_dates=['Timestamp'])
Yt = Yt.rename(columns={'Timestamp': 'time'})

In [3]:
wind_cols = {
    "10m": "Ch4_Anem_10.00m_E_Avg_m/s",
    "30m": "Ch3_Anem_30.00m_E_Avg_m/s",
    "50m": "Ch2_Anem_50.00m_E_Avg_m/s",
    "110m": "Ch1_Anem_110.00m_E_Avg_m/s"
}
stats = []
for h, col in wind_cols.items():
    series = Yt[col].dropna()
    stats.append({
        "Height": h,
        "Samples": len(series),
        "Min": series.min(),
        "Max": series.max(),
        "Mean": series.mean(),
        "Std": series.std()
    })

df_stats = pd.DataFrame(stats)
print(df_stats)

  Height  Samples     Min      Max      Mean       Std
0    10m    54339  0.0243  22.2048  5.471900  3.704020
1    30m    54339  0.0241  24.2337  6.017769  3.873758
2    50m    54339  0.0242  25.7748  6.412292  4.006668
3   110m    54339  0.0241  28.1332  7.123063  4.178140


In [4]:
total_points = len(Yt)
print("Total data points:", total_points)

for h, col in wind_cols.items():
    n = Yt[col].dropna().shape[0]
    print(f"{h}: {n} data points")

Total data points: 54339
10m: 54339 data points
30m: 54339 data points
50m: 54339 data points
110m: 54339 data points


In [3]:
Yt.head()

,time,Ch1_Anem_110.00m_E_Avg_m/s,Ch1_Anem_110.00m_E_SD_m/s,Ch1_Anem_110.00m_E_Min_m/s,Ch1_Anem_110.00m_E_Max_m/s,Ch1_Anem_110.00m_E_Gust_m/s,Ch2_Anem_50.00m_E_Avg_m/s,Ch2_Anem_50.00m_E_SD_m/s,Ch2_Anem_50.00m_E_Min_m/s,Ch2_Anem_50.00m_E_Max_m/s,...,Ch8_Vane_10.00m_N_SD_Deg,Ch8_Vane_10.00m_N_GustDir_Deg,Ch9_Analog_10.00m_N_Avg_C,Ch9_Analog_10.00m_N_SD_C,Ch9_Analog_10.00m_N_Min_C,Ch9_Analog_10.00m_N_Max_C,Ch10_Analog_10.00m_N_Avg_kpa,Ch10_Analog_10.00m_N_SD_kpa,Ch10_Analog_10.00m_N_Min_kpa,Ch10_Analog_10.00m_N_Max_kpa
0,2022-03-18 15:10:00,19.3027,1.4128,17.7532,20.5327,22.14,18.2800,1.2432,16.9740,19.7218,...,8.1590,325,7.9813,0.1875,7.6942,8.2100,87.89658,0.05752,87.87433,87.90950
1,2022-03-18 15:20:00,18.6502,1.5535,16.9952,20.8180,21.78,17.9901,2.0494,15.2208,20.0780,...,7.3742,317,8.1706,0.1444,7.9877,8.3478,87.90535,0.06745,87.89950,87.90950
2,2022-03-18 15:30:00,18.5842,1.2445,17.0272,19.9038,21.15,17.9235,1.4745,16.7717,19.4028,...,7.2353,329,7.8708,0.2128,7.5945,8.1932,87.90435,0.06738,87.89950,87.90950
3,2022-03-18 15:40:00,19.4174,1.3299,17.6112,20.6632,22.17,19.1229,1.4717,17.5193,20.2957,...,6.6146,332,7.5335,0.1685,7.3887,7.8342,87.91073,0.06977,87.89950,87.96333
4,2022-03-18 15:50:00,18.0506,2.0017,14.5360,20.2210,22.95,17.6225,1.9575,14.7990,20.2393,...,6.0053,328,7.3926,0.1183,7.2607,7.5052,87.99560,0.04206,87.94467,88.01000


In [5]:
# wind direction to sin/cos
Yt['wind_sin'] = np.sin(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))
Yt['wind_cos'] = np.cos(np.deg2rad(Yt['Ch8_Vane_10.00m_N_Avg_Deg'] % 360))

# SD turbulence columns
sd_cols = [
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s']

# Turbulence intensity TI = SD / mean
Yt['TI_110'] = Yt['Ch1_Anem_110.00m_E_SD_m/s'] / Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['TI_50']  = Yt['Ch2_Anem_50.00m_E_SD_m/s']  / Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['TI_30']  = Yt['Ch3_Anem_30.00m_E_SD_m/s']  / Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['TI_10']  = Yt['Ch4_Anem_10.00m_E_SD_m/s']  / Yt['Ch4_Anem_10.00m_E_Avg_m/s']

Yt.replace([np.inf, -np.inf], np.nan, inplace=True)
Yt.fillna(0, inplace=True)

# Gust deviation
Yt['gust_dev_110'] = Yt['Ch1_Anem_110.00m_E_Gust_m/s'] - Yt['Ch1_Anem_110.00m_E_Avg_m/s']
Yt['gust_dev_50']  = Yt['Ch2_Anem_50.00m_E_Gust_m/s']  - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['gust_dev_30']  = Yt['Ch3_Anem_30.00m_E_Gust_m/s']  - Yt['Ch3_Anem_30.00m_E_Avg_m/s']
Yt['gust_dev_10']  = Yt['Ch4_Anem_10.00m_E_Gust_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# Vertical shear
Yt['shear_110_10'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_110_50'] = Yt['Ch1_Anem_110.00m_E_Avg_m/s'] - Yt['Ch2_Anem_50.00m_E_Avg_m/s']
Yt['shear_50_10']  = Yt['Ch2_Anem_50.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']
Yt['shear_30_10']  = Yt['Ch3_Anem_30.00m_E_Avg_m/s']  - Yt['Ch4_Anem_10.00m_E_Avg_m/s']

# Temperature and pressure
Yt['temp'] = Yt['Ch9_Analog_10.00m_N_Avg_C']
Yt['pressure'] = Yt['Ch10_Analog_10.00m_N_Avg_kpa']

# Time cyclic features
Yt['minute'] = Yt['time'].dt.minute
Yt['minute_sin'] = np.sin(2 * np.pi * Yt['minute'] / 60)
Yt['minute_cos'] = np.cos(2 * np.pi * Yt['minute'] / 60)

# Base feature list
features = [
    # raw wind speeds
    'Ch4_Anem_10.00m_E_Avg_m/s',
    'Ch3_Anem_30.00m_E_Avg_m/s',
    'Ch2_Anem_50.00m_E_Avg_m/s',
    'Ch1_Anem_110.00m_E_Avg_m/s',
    # wind direction
    'wind_sin', 'wind_cos',
    # SD turbulence
    'Ch1_Anem_110.00m_E_SD_m/s',
    'Ch2_Anem_50.00m_E_SD_m/s',
    'Ch3_Anem_30.00m_E_SD_m/s',
    'Ch4_Anem_10.00m_E_SD_m/s',
    # TI
    'TI_110', 'TI_50', 'TI_30', 'TI_10',
    # gust deviation
    'gust_dev_110', 'gust_dev_50', 'gust_dev_30', 'gust_dev_10',
    # shear
    'shear_110_10', 'shear_110_50', 'shear_50_10', 'shear_30_10',
    # meteo
    'temp', 'pressure',
    # time
    'minute_sin', 'minute_cos']
target_h = "Ch4_Anem_10.00m_E_Avg_m/s"

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

def dataset_seq(data_scaled, window_size, pred_steps, target_idx):
    X, y = [], []
    n = len(data_scaled)
    for i in range(n - window_size - pred_steps):
        X.append(data_scaled[i:i + window_size, :])
        y.append(
            data_scaled[
                i + window_size : i + window_size + pred_steps,
                target_idx])
    return np.array(X), np.array(y)

def mape(true, pred):
    true = np.asarray(true)
    pred = np.asarray(pred)
    return np.mean(np.abs((true - pred) / np.maximum(np.abs(true), 1e-6))) * 100

f_list = [1, 2, 3, 4, 5, 6] 
w_list = [3,6, 12, 24]      
target = target_h[0]     
target_idx = features.index(target)

results = []
predictions = {}

df_y = Yt[['time'] + features].copy()
df_y = df_y.sort_values('time').reset_index(drop=True)

split_idx = int(0.8 * len(df_y))
train_df = df_y[features].iloc[:split_idx]
test_df  = df_y[features].iloc[split_idx:]

y_train_real = train_df[[target]].values.astype(float)
y_test_real  = test_df[[target]].values.astype(float)

scaler_X = MinMaxScaler()
train_scaled = scaler_X.fit_transform(train_df.values.astype(float))
test_scaled  = scaler_X.transform(test_df.values.astype(float))

scaler_y_real = MinMaxScaler()
scaler_y_real.fit(y_train_real)

for w in w_list:
    print(f"\n========== Window size = {w} ==========")

    for f in f_list:
        forecast_min = f * 10
        print(f"Training LSTM-only | {forecast_min} min ahead")

        start_time = time.time()
        X_train, y_train = dataset_seq(
            train_scaled, w, f, target_idx
        )
        X_test, y_test = dataset_seq(
            test_scaled, w, f, target_idx)
        scaler_y_model = MinMaxScaler()
        scaler_y_model.fit(y_train.reshape(-1, 1))

        y_train_s = scaler_y_model.transform(
            y_train.reshape(-1, 1)).reshape(-1, f)

        y_test_s = scaler_y_model.transform(
            y_test.reshape(-1, 1)).reshape(-1, f)
        model = Sequential([
            LSTM(
                f,
                input_shape=(X_train.shape[1], X_train.shape[2]),
                activation='tanh')])
        model.compile(optimizer='adam', loss='mse')
        model.fit(
            X_train, y_train_s,
            epochs=150,
            batch_size=128,
            shuffle=False,
            validation_split=0.1,
            callbacks=[
                EarlyStopping(
                    monitor='val_loss',
                    patience=5,
                    restore_best_weights=True)],
            verbose=0)

        elapsed = time.time() - start_time
        y_train_pred_s = model.predict(X_train, verbose=0)
        y_test_pred_s  = model.predict(X_test, verbose=0)

        y_train_pred_s = y_train_pred_s.reshape(-1, f)
        y_test_pred_s  = y_test_pred_s.reshape(-1, f)
        
        y_train_s_res = y_train_s.reshape(-1, f)
        y_test_s_res  = y_test_s.reshape(-1, f)

        train_pred_real = scaler_y_real.inverse_transform(
            scaler_y_model.inverse_transform(y_train_pred_s.reshape(-1, 1))
        ).reshape(-1, f)

        train_true_real = scaler_y_real.inverse_transform(
            scaler_y_model.inverse_transform(y_train_s_res.reshape(-1, 1))
        ).reshape(-1, f)
        
        test_pred_real = scaler_y_real.inverse_transform(
            scaler_y_model.inverse_transform(y_test_pred_s.reshape(-1, 1))
        ).reshape(-1, f)

        test_true_real = scaler_y_real.inverse_transform(
            scaler_y_model.inverse_transform(y_test_s_res.reshape(-1, 1))
        ).reshape(-1, f)

        y_train_true_flat = train_true_real.flatten()
        y_train_pred_flat = train_pred_real.flatten()
        y_test_true_flat  = test_true_real.flatten()
        y_test_pred_flat  = test_pred_real.flatten()

        rmse_train = np.sqrt(np.mean((y_train_true_flat - y_train_pred_flat)**2))
        rmse_test  = np.sqrt(np.mean((y_test_true_flat  - y_test_pred_flat )**2))

        r2_train = r2_score(y_train_true_flat, y_train_pred_flat)
        r2_test  = r2_score(y_test_true_flat,  y_test_pred_flat)

        mae_train = mean_absolute_error(y_train_true_flat, y_train_pred_flat)
        mae_test  = mean_absolute_error(y_test_true_flat,  y_test_pred_flat)

        mape_train = mape(y_train_true_flat, y_train_pred_flat)
        mape_test  = mape(y_test_true_flat,  y_test_pred_flat)

        print(
            f"Test RMSE={rmse_test:.4f}, "
            f"R2={r2_test:.4f}, "
            f"MAE={mae_test:.4f}, "
            f"MAPE={mape_test:.2f}%")
        
        results.append({
            "Height": target,
            "Window": w,
            "Forecast_Step": f,
            "Forecast_Min": forecast_min,
            "Train_RMSE": rmse_train,
            "Test_RMSE": rmse_test,
            "Train_R2": r2_train,
            "Test_R2": r2_test,
            "Train_MAE": mae_train,
            "Test_MAE": mae_test,
            "Train_MAPE": mape_train,
            "Test_MAPE": mape_test,
            "Time_sec": elapsed})

        predictions[(w, f, target)] = {
            "train_true_real": train_true_real.copy(),
            "train_pred_real": train_pred_real.copy(),
            "test_true_real":  test_true_real.copy(),
            "test_pred_real":  test_pred_real.copy()
        }

        K.clear_session()
        gc.collect()


========== Window size = 12 ==========
Training LSTM-only | 10 min ahead


2026-01-08 15:37:41.358107: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-01-08 15:37:41.358134: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-01-08 15:37:41.358138: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-01-08 15:37:41.358199: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-01-08 15:37:41.358416: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-01-08 15:37:42.137713: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


Test RMSE=0.8322, R2=0.9512, MAE=0.6158, MAPE=41.86%


NameError: name 'y_train_true_real' is not defined

In [ ]:
colors = ["#006699", "#b30000", "#009933",
          "#ff9900", "#660066", "#666600"]

plt.rcParams["font.size"] = 13
def plot_saved_by_w(predictions_dict, base_dir):
    for idx, ((w, f, target), data) in enumerate(predictions_dict.items()):
        w_dir = os.path.join(base_dir, f"w{w}")
        os.makedirs(w_dir, exist_ok=True)
        true_vals = data["test_true"]
        pred_vals = data["test_pred"]
        rmse = np.sqrt(np.mean((true_vals - pred_vals) ** 2))
        r2 = r2_score(true_vals, pred_vals)
        forecast_min = f * 10
        min_val = min(true_vals.min(), pred_vals.min())
        max_val = max(true_vals.max(), pred_vals.max())

        plt.figure(figsize=(7, 7))
        plt.scatter(
            true_vals,
            pred_vals,
            alpha=0.35,
            color=colors[idx % len(colors)],
            edgecolor="none")
        plt.plot(
            [min_val, max_val],
            [min_val, max_val],
            "r--",
            linewidth=2,
            label="Ideal Fit")
        plt.xlabel("True Wind Speed (m/s)")
        plt.ylabel("Predicted Wind Speed (m/s)")
        plt.title(f"{forecast_min}-Minute Ahead Prediction (w={w})")
        plt.text(
            min_val,
            max_val,
            f"$R^2 = {r2:.4f}$\nRMSE = {rmse:.4f}",
            verticalalignment="top",
            bbox=dict(facecolor="white", alpha=0.85))
        plt.grid(alpha=0.35)
        plt.legend()
        plt.tight_layout()
        plt.savefig(
            os.path.join(w_dir, f"scatter_{forecast_min}min.png"),
            dpi=300)
        plt.close()